# Chapter 5 - Classification


**Nama:** Aryo Zaki Abdillah  
**NIM:** 101032300060  
**Kelas:** TK47-02  
**Universitas:** Telkom University

> Reproduksi kode dan pendalaman teoretis dari *Practical Statistics for Data Scientists* (O'Reilly), edisi ke-2.
>
> Struktur topik mengikuti notebook Python chapter 1–7 pada repository resmi:
> https://github.com/gedeck/practical-statistics-for-data-scientists

## Tujuan

Chapter ini membahas klasifikasi (classification) sebagai metode untuk memprediksi variabel target kategorikal.
Topik yang dibahas meliputi Naive Bayes, Linear Discriminant Analysis (LDA) dan Quadratic Discriminant Analysis (QDA),
Logistic Regression, evaluasi model klasifikasi (Confusion Matrix, Precision, Recall, Specificity, ROC, AUC, Lift),
serta strategi penanganan imbalanced data.

## Imports dan Setup Data

Mengimpor pustaka Python yang dibutuhkan dan membangkitkan/memuat dataset untuk simulasi klasifikasi.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
)
from sklearn.utils import resample

np.random.seed(42)

## 1. Naive Bayes Classification

Naive Bayes menggunakan Teorema Bayes dengan asumsi independensi bersyarat antar variabel prediktor untuk menghitung probabilitas posterior dari setiap kelas.

In [ ]:
# Simulasi dataset klasifikasi pinjaman (Loan Default)
X_raw, y_raw = make_classification(
    n_samples=1000,
    n_features=5,
    n_informative=3,
    n_redundant=1,
    n_classes=2,
    weights=[0.8, 0.2],
    random_state=42
)

feature_names = ["Income", "CreditScore", "DebtRatio", "EmploymentYears", "ExistingLoans"]
df_loan = pd.DataFrame(X_raw, columns=feature_names)
df_loan["Default"] = y_raw

X_train, X_test, y_train, y_test = train_test_split(
    df_loan[feature_names], df_loan["Default"], test_size=0.3, random_state=42
)

# Naive Bayes Classifier
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)
y_prob_nb = nb_model.predict_proba(X_test)[:, 1]

print("Akurasi Naive Bayes:", accuracy_score(y_test, y_pred_nb))
print("Precision Naive Bayes:", precision_score(y_test, y_pred_nb))
print("Recall Naive Bayes:", recall_score(y_test, y_pred_nb))

## 2. Discriminant Analysis (LDA & QDA)

Linear Discriminant Analysis (LDA) dan Quadratic Discriminant Analysis (QDA) mengasumsikan variabel prediktor terdistribusi normal multivariat. LDA mengasumsikan matriks kovarians antar kelas sama, menghasilkan batas keputusan linear. QDA memungkinkan matriks kovarians berbeda antar kelas, menghasilkan batas keputusan kuadratik.

In [ ]:
# Linear Discriminant Analysis (LDA)
lda_model = LinearDiscriminantAnalysis()
lda_model.fit(X_train, y_train)
y_pred_lda = lda_model.predict(X_test)

# Quadratic Discriminant Analysis (QDA)
qda_model = QuadraticDiscriminantAnalysis()
qda_model.fit(X_train, y_train)
y_pred_qda = qda_model.predict(X_test)

print("Akurasi LDA:", accuracy_score(y_test, y_pred_lda))
print("Akurasi QDA:", accuracy_score(y_test, y_pred_qda))

## 3. Logistic Regression

Logistic Regression memodelkan probabilitas outcome menggunakan fungsi logit (log-odds):
$$
\log\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 X_1 + \dots + \beta_p X_p
$$
Model ini dipaskan menggunakan Maximum Likelihood Estimation (MLE).

In [ ]:
# Regresi Logistik dengan scikit-learn
logit_sk = LogisticRegression()
logit_sk.fit(X_train, y_train)

y_pred_logit = logit_sk.predict(X_test)
y_prob_logit = logit_sk.predict_proba(X_test)[:, 1]

print("Intercept:", logit_sk.intercept_[0])
for col, coef in zip(feature_names, logit_sk.coef_[0]):
    print(f"{col}: {coef:.4f} (Odds Ratio: {np.exp(coef):.4f})")

## 4. Logistic Regression dengan statsmodels

statsmodels memberikan ringkasan statistik yang lebih rinci, termasuk z-statistic, p-value, dan interval kepercayaan untuk setiap koefisien.

In [ ]:
X_train_sm = sm.add_constant(X_train)
logit_sm = sm.Logit(y_train, X_train_sm).fit()

print(logit_sm.summary())

## 5. Evaluating Classification Models

Evaluasi model klasifikasi tidak hanya menggunakan akurasi, melainkan juga Confusion Matrix, Precision, Recall (Sensitivity), Specificity, ROC Curve, AUC, dan Lift Chart.

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_logit)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Non-Default", "Default"])

plt.figure(figsize=(5, 4))
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix - Logistic Regression")
plt.show()

# Kurva ROC dan AUC
fpr, tpr, thresholds = roc_curve(y_test, y_prob_logit)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlabel("False Positive Rate (1 - Specificity)")
plt.ylabel("True Positive Rate (Recall)")
plt.title("Receiver Operating Characteristic (ROC)")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

## 6. Strategies for Imbalanced Data

Pada kasus data tidak seimbang (ketika kelas minoritas jauh lebih sedikit), prediksi sederhana cenderung memprediksi kelas mayoritas. Strategi seperti resample (oversampling / undersampling) atau pembobotan kelas (class weight) digunakan untuk menyeimbangkan performa.

In [ ]:
# Menggunakan Class Weight Balanced pada Logistic Regression
logit_balanced = LogisticRegression(class_weight="balanced")
logit_balanced.fit(X_train, y_train)
y_pred_bal = logit_balanced.predict(X_test)

print("=== Logistic Regression Standar ===")
print("Precision:", precision_score(y_test, y_pred_logit))
print("Recall:", recall_score(y_test, y_pred_logit))

print("\n=== Logistic Regression Class Balanced ===")
print("Precision:", precision_score(y_test, y_pred_bal))
print("Recall:", recall_score(y_test, y_pred_bal))

## Interpretasi

Dari analisis klasifikasi pada Chapter 5, dapat disimpulkan bahwa Logistic Regression memberi keunggulan dalam hal interpretabilitas melalui odds ratio. Naive Bayes sangat efisien dan cocok sebagai baseline model, sedangkan LDA dan QDA bekerja optimal jika asumsi distribusi normal multivariat terpenuhi.

Pada kasus imbalanced data, akurasi umum bisa menyesatkan sehingga pengujian harus berfokus pada Precision, Recall, F1-Score, dan AUC-ROC.

## Kesimpulan Chapter 5

Klasifikasi menyediakan kerangka kerja statistik untuk memprediksi kategori/kelompok outcome. Evaluasi model wajib disesuaikan dengan konteks bisnis (misalnya meminimalisir False Negative pada prediksi fraud atau kredit macet) menggunakan metrik yang tepat dan penanganan imbalanced data yang sesuai.